In [1]:
!pip install transformers peft bitsandbytes accelerate trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.2 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requir

In [2]:
import torch
import json
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, TaskType
from trl import RewardTrainer, RewardConfig

print("Libraries imported!")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Libraries imported!
GPU: Tesla T4


In [4]:
sft_model_path = "/kaggle/input/datasets/rlhf0226/sft-model"

tokenizer = AutoTokenizer.from_pretrained(sft_model_path)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded!")

Tokenizer loaded!


In [5]:
data_path = "/kaggle/input/datasets/rlhf0226/reward-data"

def load_jsonl(file):
    with open(file) as f:
        return [json.loads(line) for line in f]

data = load_jsonl(data_path + "/reward_train (1).jsonl")
print(f"Loaded {len(data)} samples")
print(f"Columns: {list(data[0].keys())}")
print(f"\nSample:")
print(data[0])

Loaded 1927 samples
Columns: ['prompt', 'chosen', 'rejected']

Sample:
{'prompt': 'Can you describe why a drought does not end when it rains', 'chosen': "A drought is usually defined as a long period of time with a shortage of water. While rain is necessary to replenish our water supplies, it isn’t a very reliable way of providing water. This can be attributed to a lack of water infrastructure and advanced weather forecasting. Humans have only a limited ability to predict the timing and amount of rainfall.\nOkay. But you didn't answer my question, you simply described a drought Sorry, I'm not sure what you mean. The sentence structure in your question doesn’t fit with what I’m used to. I think you may have been looking for me to explain something else, like why rain doesn’t end a drought? The answer is that it doesn’t end a drought because the amount of water that falls to the ground in a rain event is often not enough to fill the gap in the groundwater. The drought is either not truly

In [13]:
# Cell 5 — No manual tokenization needed, just format raw text
def format_dataset(example):
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_dataset)

print(f"Dataset ready!")
print(f"Columns: {dataset.column_names}")
print(f"Size: {len(dataset)} samples")

Map:   0%|          | 0/1927 [00:00<?, ? examples/s]

Dataset ready!
Columns: ['prompt', 'chosen', 'rejected']
Size: 1927 samples


In [14]:
# Cell 6 — Split stays the same
split      = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")

Train: 1734 samples
Val:   193 samples


In [15]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading model... (2-3 mins)")
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!")

Loading model... (2-3 mins)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


In [16]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

print("LoRA config ready!")

LoRA config ready!


In [17]:
training_args = RewardConfig(
    output_dir="/kaggle/working/reward_model",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1.41e-5,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=False,
    bf16=True,
    report_to="none",
    max_length=512
)

# Fix to prevent error
training_args.center_rewards_coefficient = None

print("Training arguments set!")

Training arguments set!


In [18]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
     processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config
)

print("Starting Reward Model Training...")
print("Watch for rewards/chosen going UP and rewards/rejected going DOWN!\n")

trainer.train()

print("Reward Model Training complete!")

Adding EOS to train dataset:   0%|          | 0/1734 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1734 [00:00<?, ? examples/s]

Filtering train >512 tokens:   0%|          | 0/1734 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Filtering eval >512 tokens:   0%|          | 0/193 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting Reward Model Training...
Watch for rewards/chosen going UP and rewards/rejected going DOWN!



Step,Training Loss,Validation Loss,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
50,0.973092,1.021711,265075.000000,3.110054,6.089462,8.870924,0.554348,0.004119
100,1.073030,1.014910,523448.000000,3.163213,6.108069,8.864130,0.570652,0.011273


Reward Model Training complete!


In [24]:
print("Evaluating Reward Model...")
print("=" * 60)

model.eval()
correct = 0
total = min(200, len(val_data))

for i in range(total):
    sample = val_data[i]

    chosen_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['chosen']}"
    rejected_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['rejected']}"

    chosen_inputs = tokenizer(
        chosen_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    rejected_inputs = tokenizer(
        rejected_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        chosen_score = model(**chosen_inputs).logits[0].item()
        rejected_score = model(**rejected_inputs).logits[0].item()

    is_correct = chosen_score > rejected_score

    if is_correct:
        correct += 1

    if i < 5 or i >= total - 3:
        status = "✅" if is_correct else "❌"
        print(
            f"Row {i+1:3d} | "
            f"chosen: {chosen_score:7.3f} | "
            f"rejected: {rejected_score:7.3f} | "
            f"correct: {str(is_correct):<5} {status}"
        )
    elif i == 5:
        print("...")

accuracy = correct / total * 100

print("=" * 60)
print(f"\nFinal Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

Evaluating Reward Model...
Row   1 | chosen:   7.750 | rejected:   8.938 | correct: False ❌
Row   2 | chosen:   7.625 | rejected:   6.656 | correct: True  ✅
Row   3 | chosen:   7.969 | rejected:   7.406 | correct: True  ✅
Row   4 | chosen:   5.438 | rejected:   7.156 | correct: False ❌
Row   5 | chosen:   6.781 | rejected:   5.344 | correct: True  ✅
...
Row 191 | chosen:   5.375 | rejected:   5.469 | correct: False ❌
Row 192 | chosen:   8.500 | rejected:  10.000 | correct: False ❌
Row 193 | chosen:   6.469 | rejected:   7.938 | correct: False ❌

Final Accuracy: 49.7% (96/193 correct)


In [20]:
trainer.save_model("/kaggle/working/reward_model")
tokenizer.save_pretrained("/kaggle/working/reward_model")

print("Reward model saved!")
print("Location: /kaggle/working/reward_model")

Reward model saved!
Location: /kaggle/working/reward_model


In [25]:
def get_reward_score(prompt, response):
    text = f"\n\nHuman: {prompt}\n\nAssistant: {response}"
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        score = model(**inputs).logits[0].item()
    return score

# Test it!
prompt       = "How do I stay healthy?"
good_response = "Exercise daily, eat balanced meals, sleep 7-8 hours!"
bad_response  = "I don't know just try stuff"

good_score = get_reward_score(prompt, good_response)
bad_score  = get_reward_score(prompt, bad_response)

print("Reward Scores:")
print(f"Good response: {good_score:.4f}")
print(f"Bad response:  {bad_score:.4f}")

if good_score > bad_score:
    print("\nReward model working correctly!")
else:
    print("\nReward model needs more training")

Reward Scores:
Good response: 7.3125
Bad response:  5.7812

Reward model working correctly!


In [26]:
import shutil

shutil.make_archive(
    "/kaggle/working/reward_model",
    "zip",
    "/kaggle/working/reward_model"
)

print("ZIP created!")

ZIP created!


In [27]:
print(type(model))
print(type(trainer.model))

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForSequenceClassification'>
<class 'peft.peft_model.PeftModelForSequenceClassification'>


In [28]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 1.0174157619476318, 'eval_runtime': 187.7491, 'eval_samples_per_second': 0.943, 'eval_steps_per_second': 0.123, 'eval_num_tokens': 539969.0, 'eval_min_reward': 3.1645720108695654, 'eval_mean_reward': 6.109215777853261, 'eval_max_reward': 8.858695652173912, 'eval_accuracy': 0.5597826086956522, 'eval_margin': 0.0035453464673913045, 'epoch': 1.0}


In [29]:
model = trainer.model

In [30]:
print("Evaluating Reward Model...")
print("=" * 60)

model.eval()
correct = 0
total = min(200, len(val_data))

for i in range(total):
    sample = val_data[i]

    chosen_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['chosen']}"
    rejected_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['rejected']}"

    chosen_inputs = tokenizer(
        chosen_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    rejected_inputs = tokenizer(
        rejected_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        chosen_score = model(**chosen_inputs).logits[0].item()
        rejected_score = model(**rejected_inputs).logits[0].item()

    is_correct = chosen_score > rejected_score

    if is_correct:
        correct += 1

    if i < 5 or i >= total - 3:
        status = "✅" if is_correct else "❌"
        print(
            f"Row {i+1:3d} | "
            f"chosen: {chosen_score:7.3f} | "
            f"rejected: {rejected_score:7.3f} | "
            f"correct: {str(is_correct):<5} {status}"
        )
    elif i == 5:
        print("...")

accuracy = correct / total * 100

print("=" * 60)
print(f"\nFinal Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

Evaluating Reward Model...
Row   1 | chosen:   7.750 | rejected:   8.938 | correct: False ❌
Row   2 | chosen:   7.625 | rejected:   6.656 | correct: True  ✅
Row   3 | chosen:   7.969 | rejected:   7.406 | correct: True  ✅
Row   4 | chosen:   5.438 | rejected:   7.156 | correct: False ❌
Row   5 | chosen:   6.781 | rejected:   5.344 | correct: True  ✅
...
Row 191 | chosen:   5.375 | rejected:   5.469 | correct: False ❌
Row 192 | chosen:   8.500 | rejected:  10.000 | correct: False ❌
Row 193 | chosen:   6.469 | rejected:   7.938 | correct: False ❌

Final Accuracy: 49.7% (96/193 correct)
